# AI vs Real Image Detection

This notebook trains and evaluates a deep learning model for distinguishing **AI-generated images** from **real images**. The project uses **EfficientNet-B0**, transfer learning, fine-tuning, evaluation metrics, and Gradio deployment preparation.

## Workflow
1. Environment setup
2. Dataset preparation
3. Data loading and preprocessing
4. Model comparison summary
5. EfficientNet-B0 training and fine-tuning
6. Evaluation and visualization
7. Gradio deployment file preparation


## 1. Environment Setup


In [ ]:
import os
import random
import shutil
import time
import copy
import zipfile
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report


In [ ]:
SEED = 42

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


## 2. Google Drive Connection
Use this cell only in Google Colab.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    print("Google Drive mount skipped. This environment may not be Google Colab.")


## 3. Dataset Preparation

The dataset is expected as a ZIP file containing `train`, `val`, and `test` folders. Each split should contain class folders such as `FAKE` and `REAL`.

```text
Islenmis_Veri/
├── train/
│   ├── FAKE/
│   └── REAL/
├── val/
│   ├── FAKE/
│   └── REAL/
└── test/
    ├── FAKE/
    └── REAL/
```


In [ ]:
DRIVE_ZIP_PATH = "/content/drive/MyDrive/veri.zip"
RAW_DATA_DIR = "/content/raw_dataset"
PROCESSED_DATA_DIR = "/content/Islenmis_Veri"
IMAGE_SIZE = (224, 224)


def prepare_dataset_from_zip(zip_path: str = DRIVE_ZIP_PATH,
                             raw_dir: str = RAW_DATA_DIR,
                             processed_dir: str = PROCESSED_DATA_DIR,
                             image_size: tuple = IMAGE_SIZE):
    # Extract dataset ZIP, convert images to RGB, resize images, and save a clean folder structure.
    if os.path.exists(raw_dir):
        shutil.rmtree(raw_dir)
    if os.path.exists(processed_dir):
        shutil.rmtree(processed_dir)

    os.makedirs(raw_dir, exist_ok=True)

    if not os.path.exists(zip_path):
        raise FileNotFoundError(f"Dataset ZIP not found: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(raw_dir)

    root_dir = raw_dir
    contents = os.listdir(raw_dir)
    if len(contents) == 1 and os.path.isdir(os.path.join(raw_dir, contents[0])):
        root_dir = os.path.join(raw_dir, contents[0])

    for split in ["train", "val", "test"]:
        split_dir = os.path.join(root_dir, split)
        if not os.path.exists(split_dir):
            print(f"Warning: '{split}' folder was not found. Skipping this split.")
            continue

        class_folders = [d for d in os.listdir(split_dir) if os.path.isdir(os.path.join(split_dir, d))]
        for class_name in class_folders:
            source_class_dir = os.path.join(split_dir, class_name)
            target_class_dir = os.path.join(processed_dir, split, class_name)
            os.makedirs(target_class_dir, exist_ok=True)

            image_files = [
                f for f in os.listdir(source_class_dir)
                if f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
            ]

            for idx, file_name in enumerate(tqdm(image_files, desc=f"{split}/{class_name}"), start=1):
                source_path = os.path.join(source_class_dir, file_name)
                target_path = os.path.join(target_class_dir, f"image_{idx:06d}.jpg")

                try:
                    with Image.open(source_path).convert("RGB") as img:
                        img = img.resize(image_size, Image.Resampling.LANCZOS)
                        img.save(target_path, quality=95)
                except Exception as exc:
                    print(f"Skipped file: {source_path} | Error: {exc}")

    print(f"Dataset preparation completed: {processed_dir}")


# Uncomment if preprocessing from ZIP is needed.
# prepare_dataset_from_zip()


## 4. Data Loading and Preprocessing


In [ ]:
DATA_DIR = "/content/Islenmis_Veri"
BATCH_SIZE = 32
NUM_WORKERS = 2

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

data_transforms = {
    "train": train_transform,
    "val": val_test_transform,
    "test": val_test_transform
}

image_datasets = {
    split: datasets.ImageFolder(os.path.join(DATA_DIR, split), data_transforms[split])
    for split in ["train", "val", "test"]
}

dataloaders = {
    "train": DataLoader(image_datasets["train"], batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS),
    "val": DataLoader(image_datasets["val"], batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS),
    "test": DataLoader(image_datasets["test"], batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
}

dataset_sizes = {split: len(image_datasets[split]) for split in ["train", "val", "test"]}
class_names = image_datasets["train"].classes

print("Class names:", class_names)
print("Class indices:", image_datasets["train"].class_to_idx)
print("Dataset sizes:", dataset_sizes)


## 5. Model Comparison Summary

Preliminary experiments showed that EfficientNet-B0 achieved the best balance between accuracy and training time.

| Model | Accuracy | Training Time (sec) |
|---|---:|---:|
| ResNet50 | 93.45% | 2618 |
| DenseNet121 | 96.01% | 2693 |
| EfficientNet-B0 | 97.23% | 1330 |

Therefore, EfficientNet-B0 was selected as the final model architecture.


## 6. EfficientNet-B0 Model Definition


In [ ]:
def create_efficientnet_b0(num_classes: int = 2, freeze_features: bool = True):
    # Create EfficientNet-B0 for binary image classification.
    weights = models.EfficientNet_B0_Weights.DEFAULT
    model = models.efficientnet_b0(weights=weights)

    if freeze_features:
        for param in model.features.parameters():
            param.requires_grad = False

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model.to(device)

model = create_efficientnet_b0(num_classes=len(class_names), freeze_features=True)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)
print(model.classifier)


## 7. Training Function


In [ ]:
def train_model(model, criterion, optimizer, dataloaders, device, num_epochs=5,
                model_path="best_efficientnet_b0.pth"):
    # Train model and save the best validation checkpoint.
    start_time = time.time()
    best_model_weights = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(num_epochs):
        print(f"Epoch {epoch + 1}/{num_epochs}")
        print("-" * 40)

        for phase in ["train", "val"]:
            model.train() if phase == "train" else model.eval()
            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels).item()

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects / len(dataloaders[phase].dataset)

            history[f"{phase}_loss"].append(epoch_loss)
            history[f"{phase}_acc"].append(epoch_acc)

            print(f"{phase.upper()} Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")

            if phase == "val" and epoch_acc > best_val_acc:
                best_val_acc = epoch_acc
                best_model_weights = copy.deepcopy(model.state_dict())
                torch.save(model.state_dict(), model_path)

        print()

    elapsed = time.time() - start_time
    print(f"Training completed in {elapsed // 60:.0f} min {elapsed % 60:.0f} sec")
    print(f"Best validation accuracy: {best_val_acc:.4f}")

    model.load_state_dict(best_model_weights)
    return model, history


## 8. Initial Training


In [ ]:
model, history_initial = train_model(
    model=model,
    criterion=criterion,
    optimizer=optimizer,
    dataloaders=dataloaders,
    device=device,
    num_epochs=5,
    model_path="best_efficientnet_b0_initial.pth"
)


## 9. Fine-Tuning


In [ ]:
for param in model.parameters():
    param.requires_grad = True

optimizer_ft = optim.Adam(model.parameters(), lr=0.0001)

model, history_ft = train_model(
    model=model,
    criterion=criterion,
    optimizer=optimizer_ft,
    dataloaders=dataloaders,
    device=device,
    num_epochs=5,
    model_path="best_efficientnet_b0_finetuned.pth"
)


## 10. Save and Load Model


In [ ]:
MODEL_OUTPUT_PATH = "best_model.pt"
torch.save(model.state_dict(), MODEL_OUTPUT_PATH)
print(f"Model saved: {MODEL_OUTPUT_PATH}")

DRIVE_MODEL_DIR = "/content/drive/MyDrive/BitirmeProjesi"
if os.path.exists("/content/drive/MyDrive"):
    os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
    shutil.copy(MODEL_OUTPUT_PATH, os.path.join(DRIVE_MODEL_DIR, MODEL_OUTPUT_PATH))
    print(f"Model copied to: {DRIVE_MODEL_DIR}")


In [ ]:
def load_model(model_path: str, num_classes: int = 2):
    model = create_efficientnet_b0(num_classes=num_classes, freeze_features=False)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model

model = load_model(MODEL_OUTPUT_PATH, num_classes=len(class_names))
print("Model loaded successfully.")


## 11. Test Evaluation


In [ ]:
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in dataloaders["test"]:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(cm)

print("
Classification Report:")
print(classification_report(all_labels, all_preds, target_names=class_names))


## 12. Training Curves


In [ ]:
def plot_training_curves(history, title_prefix="Fine-Tuning"):
    plt.figure(figsize=(8, 5))
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix} Loss Graph")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("fine_tuning_loss.png", dpi=300)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(history["train_acc"], label="Train Accuracy")
    plt.plot(history["val_acc"], label="Val Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{title_prefix} Accuracy Graph")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig("fine_tuning_accuracy.png", dpi=300)
    plt.show()

plot_training_curves(history_ft, title_prefix="Fine-Tuning")


## 13. Confusion Matrix Visualization


In [ ]:
def plot_confusion_matrix(cm, labels):
    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_title("Confusion Matrix")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center")

    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=300)
    plt.show()

plot_confusion_matrix(cm, class_names)


## 14. Single Image Prediction


In [ ]:
def predict_image(image_path: str, model, class_names):
    image = Image.open(image_path).convert("RGB")
    image_tensor = val_test_transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted_idx = torch.max(probabilities, 1)

    return {
        "prediction": class_names[predicted_idx.item()],
        "confidence": float(confidence.item()),
        "probabilities": {
            class_names[i]: float(probabilities[0][i].item())
            for i in range(len(class_names))
        }
    }

# Example:
# predict_image("example.jpg", model, class_names)


## 15. Gradio App File


In [ ]:
app_code = 'import torch\nimport torch.nn as nn\nimport gradio as gr\nfrom PIL import Image\nfrom torchvision import models, transforms\n\nMODEL_PATH = "models/best_model.pt"\nCLASS_NAMES = ["FAKE", "REAL"]\nDEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n\ntransform = transforms.Compose([\n    transforms.Resize((224, 224)),\n    transforms.ToTensor(),\n    transforms.Normalize([0.485, 0.456, 0.406],\n                         [0.229, 0.224, 0.225])\n])\n\ndef create_model():\n    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)\n    in_features = model.classifier[1].in_features\n    model.classifier[1] = nn.Linear(in_features, len(CLASS_NAMES))\n    state_dict = torch.load(MODEL_PATH, map_location=DEVICE)\n    model.load_state_dict(state_dict)\n    model.to(DEVICE)\n    model.eval()\n    return model\n\nmodel = create_model()\n\ndef predict(image, uncertainty_threshold=0.75):\n    if image is None:\n        return {}, "Please upload an image."\n\n    image = image.convert("RGB")\n    image_tensor = transform(image).unsqueeze(0).to(DEVICE)\n\n    with torch.no_grad():\n        outputs = model(image_tensor)\n        probabilities = torch.softmax(outputs, dim=1)[0]\n\n    scores = {CLASS_NAMES[i]: float(probabilities[i]) for i in range(len(CLASS_NAMES))}\n    predicted_class = max(scores, key=scores.get)\n    confidence = scores[predicted_class]\n\n    if confidence < uncertainty_threshold:\n        detail = f"Prediction: UNCERTAIN | Confidence: {confidence:.2%}"\n    else:\n        detail = f"Prediction: {predicted_class} | Confidence: {confidence:.2%}"\n\n    return scores, detail\n\nwith gr.Blocks(title="AI vs Real Image Detection") as demo:\n    gr.Markdown("# Yapay mı Gerçek mi Görsel Tespit Sistemi")\n    gr.Markdown("Bir görsel yükleyin. Model görselin FAKE mi REAL mi olduğunu tahmin eder.")\n\n    with gr.Row():\n        with gr.Column():\n            image_input = gr.Image(type="pil", label="Görsel Yükle")\n            threshold = gr.Slider(0.50, 0.95, value=0.75, step=0.01, label="Kararsızlık Eşiği")\n            submit_btn = gr.Button("Submit")\n        with gr.Column():\n            label_output = gr.Label(label="Sınıf Olasılıkları")\n            text_output = gr.Textbox(label="Tahmin Detayı")\n\n    submit_btn.click(\n        fn=predict,\n        inputs=[image_input, threshold],\n        outputs=[label_output, text_output]\n    )\n\ndemo.launch()'

with open("app.py", "w", encoding="utf-8") as file:
    file.write(app_code)

print("app.py created successfully.")


## 16. Notes for GitHub

Before uploading to GitHub:

- Clear notebook outputs.
- Do not upload private Drive paths, tokens, or large datasets.
- Store large model files with Git LFS or provide a download link.
- Add `README.md`, `requirements.txt`, and `.gitignore`.
